<a href="https://colab.research.google.com/github/JorgeCaldeira/Beware-the-dog/blob/main/DL_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##TODO
Please add max 2 pages to the report of Milestone 0 with:

*   data pipeline, including preprocessing
baseline description:

    *  model name / architecture depth;
    *  input shape;
    *  loss;
    *  optimiser;
    *  training budget (epochs, LR, batch, GPU time)


* preliminary results and training curves
* known issues and next steps, including a detailed plan for the rest of the experiments






By the first milestone, it is expected that you have the data downloaded and ready to be used, the models clearly identified and a skeleton code to start working on.

This time, the document should be 3 pages max and it can include whatever is still relevant from the proposal. It should have:
* Title, group members
* Introduction: this section introduces your problem, and the overall plan for approaching your problem
* Problem statement: Describe your problem precisely specifying the dataset to be used, expected results and evaluation
* Technical Approach: Describe the methods you intend to apply to solve the given problem
* Figure describing the approach / methodology.

Get inspiration from the papers that you’re reading for your project. Some examples:
figure 1 of this paper or similar
check the posters here or go check LASIGE posters in C6 floor 2.
Describe the experiments you are planning to do and preliminary results if you have them.



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
import os

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
for root, dirs, files in os.walk('/content/drive'):
    if 'dog-breed-identification' in root:
        print(root)

/content/drive/.shortcut-targets-by-id/1OBtF3Z4ZtdyduyQMzRNMvGbRBtun6fZc/AP Projeto/dog-breed-identification


KeyboardInterrupt: 

In [4]:
import os
os.chdir('/content/drive/.shortcut-targets-by-id/1OBtF3Z4ZtdyduyQMzRNMvGbRBtun6fZc/AP Projeto/dog-breed-identification')

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

df = pd.read_csv('labels.csv')
breeds = sorted(df['breed'].unique())
breed_to_idx = {breed: i for i, breed in enumerate(breeds)}

class DogDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx, 0]
        label_name = self.df.iloc[idx, 1]
        label = breed_to_idx[label_name]

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
transform_baseline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_augmented = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
def get_model():
    model = models.resnet50(pretrained=True)

    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, 120)

    return model

model = get_model()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.0001)

In [ ]:
batch_size = 32

train_dataset = DogDataset(df, 'train/', transform=transform_augmented)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Resetar gradientes
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels) # Cross-Entropy Loss

        # Backward pass e otimização
        loss.backward()
        optimizer.step()

        # Estatísticas
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100. * correct / total

In [ ]:
epochs = 10
train_losses = []
train_accs = []

for epoch in range(epochs):
    loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    train_accs.append(acc)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f}, Acc: {acc:.2f}%")

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Training Accuracy')
plt.title('Accuracy Curve')
plt.legend()
plt.show()